# CNN可視化技術

深度學習模型常被視為「黑盒子」，但通過可視化技術，我們可以理解CNN學到了什麼，以及它如何做出決策。

## 本章內容

1. **特徵圖可視化** - 查看每層學到的特徵
2. **卷積核可視化** - 理解濾波器的作用
3. **Grad-CAM** - 類激活映射，找到模型關注的區域
4. **激活最大化** - 生成最大化特定神經元激活的圖像
5. **特徵空間可視化** - 使用t-SNE/UMAP降維可視化
6. **訓練過程可視化** - 損失曲線、準確率曲線

## 學習目標

- 理解CNN各層學習到的特徵
- 掌握多種可視化技術的實現
- 學會調試和改進模型
- 提升模型的可解釋性

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import DataLoader
import torchvision
from torchvision import transforms, models
import matplotlib.pyplot as plt
import numpy as np
from PIL import Image
import cv2
from sklearn.manifold import TSNE
import seaborn as sns

# 設置隨機種子
torch.manual_seed(42)
np.random.seed(42)

# 設備配置
device = torch.device('cuda' if torch.cuda.is_available() else 
                     'mps' if torch.backends.mps.is_available() else 'cpu')
print(f'使用設備: {device}')

# 設置matplotlib
plt.rcParams['figure.figsize'] = (12, 8)
plt.rcParams['font.size'] = 10

## 1. 特徵圖可視化

特徵圖(Feature Maps)顯示了卷積層對輸入的響應。通過可視化特徵圖，我們可以：
- 理解每層提取了什麼樣的特徵
- 觀察特徵的層次性（從簡單到複雜）
- 診斷模型問題

### 特徵層次

- **淺層**：邊緣、顏色、紋理
- **中層**：形狀、局部模式
- **深層**：高級語義特徵、物體部件

In [ ]:
# 定義一個簡單的CNN用於演示
class SimpleCNN(nn.Module):
    def __init__(self):
        super(SimpleCNN, self).__init__()
        self.conv1 = nn.Conv2d(1, 16, kernel_size=3, padding=1)
        self.conv2 = nn.Conv2d(16, 32, kernel_size=3, padding=1)
        self.conv3 = nn.Conv2d(32, 64, kernel_size=3, padding=1)
        self.pool = nn.MaxPool2d(2, 2)
        self.fc1 = nn.Linear(64 * 3 * 3, 128)
        self.fc2 = nn.Linear(128, 10)
        
    def forward(self, x):
        x = self.pool(F.relu(self.conv1(x)))
        x = self.pool(F.relu(self.conv2(x)))
        x = self.pool(F.relu(self.conv3(x)))
        x = x.view(x.size(0), -1)
        x = F.relu(self.fc1(x))
        x = self.fc2(x)
        return x

# 創建模型
model = SimpleCNN().to(device)
print(model)

In [ ]:
# 載入一張測試圖像
from torchvision.datasets import FashionMNIST

dataset = FashionMNIST(root='./data', train=True, download=True, 
                       transform=transforms.ToTensor())
image, label = dataset[0]
image = image.unsqueeze(0).to(device)  # 添加批次維度

print(f"圖像形狀: {image.shape}")
print(f"標籤: {label}")

# 顯示原始圖像
plt.imshow(image.cpu().squeeze(), cmap='gray')
plt.title(f'原始圖像 (標籤: {label})')
plt.axis('off')
plt.show()

In [ ]:
def visualize_feature_maps(model, image, layer_names=None):
    """
    可視化卷積層的特徵圖
    
    Args:
        model: CNN模型
        image: 輸入圖像
        layer_names: 要可視化的層名稱列表
    """
    model.eval()
    
    # 註冊hook來獲取中間層輸出
    activations = {}
    
    def get_activation(name):
        def hook(model, input, output):
            activations[name] = output.detach()
        return hook
    
    # 為每個卷積層註冊hook
    hooks = []
    for name, layer in model.named_modules():
        if isinstance(layer, nn.Conv2d):
            hooks.append(layer.register_forward_hook(get_activation(name)))
    
    # 前向傳播
    with torch.no_grad():
        output = model(image)
    
    # 移除hooks
    for hook in hooks:
        hook.remove()
    
    # 可視化每層的特徵圖
    for layer_name, activation in activations.items():
        print(f"\n{layer_name}: {activation.shape}")
        
        # 獲取特徵圖數量
        n_features = activation.shape[1]
        
        # 顯示前16個特徵圖（如果有的話）
        n_display = min(16, n_features)
        n_cols = 4
        n_rows = (n_display + n_cols - 1) // n_cols
        
        fig, axes = plt.subplots(n_rows, n_cols, figsize=(12, 3*n_rows))
        axes = axes.ravel() if n_display > 1 else [axes]
        
        for i in range(n_display):
            feature_map = activation[0, i].cpu().numpy()
            axes[i].imshow(feature_map, cmap='viridis')
            axes[i].set_title(f'特徵圖 {i+1}')
            axes[i].axis('off')
        
        # 隱藏多餘的子圖
        for i in range(n_display, len(axes)):
            axes[i].axis('off')
        
        plt.suptitle(f'{layer_name} 的特徵圖', fontsize=14, y=1.0)
        plt.tight_layout()
        plt.show()

# 可視化特徵圖
visualize_feature_maps(model, image)

## 2. 卷積核可視化

卷積核(Filters/Kernels)是CNN學習到的參數，可視化卷積核可以幫助我們理解：
- 網路在尋找什麼樣的模式
- 不同層關注的特徵
- 是否存在死神經元（全為0的卷積核）

In [ ]:
def visualize_filters(model, layer_name='conv1'):
    """
    可視化指定卷積層的卷積核
    
    Args:
        model: CNN模型
        layer_name: 卷積層的名稱
    """
    # 獲取指定層的權重
    for name, param in model.named_parameters():
        if layer_name in name and 'weight' in name:
            filters = param.data.cpu().numpy()
            print(f"{name} 的形狀: {filters.shape}")
            print(f"(輸出通道數, 輸入通道數, 高度, 寬度)")
            
            # 歸一化到[0, 1]
            filters = (filters - filters.min()) / (filters.max() - filters.min())
            
            # 顯示卷積核
            n_filters = filters.shape[0]
            n_display = min(16, n_filters)
            n_cols = 4
            n_rows = (n_display + n_cols - 1) // n_cols
            
            fig, axes = plt.subplots(n_rows, n_cols, figsize=(12, 3*n_rows))
            axes = axes.ravel() if n_display > 1 else [axes]
            
            for i in range(n_display):
                # 對於多通道輸入，取第一個通道或平均
                if filters.shape[1] == 1:
                    filter_img = filters[i, 0]
                else:
                    filter_img = filters[i].mean(axis=0)
                
                axes[i].imshow(filter_img, cmap='viridis')
                axes[i].set_title(f'卷積核 {i+1}')
                axes[i].axis('off')
            
            # 隱藏多餘的子圖
            for i in range(n_display, len(axes)):
                axes[i].axis('off')
            
            plt.suptitle(f'{name} 的卷積核', fontsize=14, y=1.0)
            plt.tight_layout()
            plt.show()
            break

# 可視化第一層的卷積核
visualize_filters(model, 'conv1')

## 3. Grad-CAM (Gradient-weighted Class Activation Mapping)

Grad-CAM是一種強大的可視化技術，可以：
- 顯示模型在做決策時關注圖像的哪些區域
- 提供模型預測的視覺解釋
- 幫助調試和改進模型

### 原理

1. 對目標類別計算梯度
2. 對特徵圖的梯度進行全局平均池化，得到權重
3. 使用權重對特徵圖進行加權求和
4. 應用ReLU獲得熱力圖

$$L_{Grad-CAM}^c = ReLU\left(\sum_k \alpha_k^c A^k\right)$$

其中 $\alpha_k^c = \frac{1}{Z}\sum_i\sum_j \frac{\partial y^c}{\partial A_{ij}^k}$

In [ ]:
class GradCAM:
    """Grad-CAM實現"""
    
    def __init__(self, model, target_layer):
        """
        Args:
            model: CNN模型
            target_layer: 目標卷積層（通常是最後一個卷積層）
        """
        self.model = model
        self.target_layer = target_layer
        self.gradients = None
        self.activations = None
        
        # 註冊hooks
        self.target_layer.register_forward_hook(self.save_activation)
        self.target_layer.register_backward_hook(self.save_gradient)
    
    def save_activation(self, module, input, output):
        """保存前向傳播的激活"""
        self.activations = output.detach()
    
    def save_gradient(self, module, grad_input, grad_output):
        """保存反向傳播的梯度"""
        self.gradients = grad_output[0].detach()
    
    def __call__(self, x, class_idx=None):
        """
        生成Grad-CAM熱力圖
        
        Args:
            x: 輸入圖像
            class_idx: 目標類別索引（None表示預測類別）
        """
        # 前向傳播
        output = self.model(x)
        
        # 如果沒有指定類別，使用預測類別
        if class_idx is None:
            class_idx = output.argmax(dim=1).item()
        
        # 清零梯度
        self.model.zero_grad()
        
        # 反向傳播
        one_hot = torch.zeros_like(output)
        one_hot[0, class_idx] = 1
        output.backward(gradient=one_hot, retain_graph=True)
        
        # 計算權重：對梯度進行全局平均池化
        weights = self.gradients.mean(dim=(2, 3), keepdim=True)
        
        # 加權求和
        cam = (weights * self.activations).sum(dim=1, keepdim=True)
        
        # 應用ReLU
        cam = F.relu(cam)
        
        # 上採樣到輸入大小
        cam = F.interpolate(cam, size=x.shape[2:], mode='bilinear', align_corners=False)
        
        # 歸一化到[0, 1]
        cam = cam - cam.min()
        cam = cam / (cam.max() + 1e-8)
        
        return cam.squeeze().cpu().numpy(), class_idx

def show_cam_on_image(img, cam, alpha=0.5):
    """
    將Grad-CAM熱力圖疊加到原始圖像上
    
    Args:
        img: 原始圖像 (numpy array)
        cam: CAM熱力圖 (numpy array)
        alpha: 透明度
    """
    # 將CAM轉換為彩色熱力圖
    heatmap = cv2.applyColorMap(np.uint8(255 * cam), cv2.COLORMAP_JET)
    heatmap = cv2.cvtColor(heatmap, cv2.COLOR_BGR2RGB)
    heatmap = heatmap / 255.0
    
    # 將圖像轉換為3通道
    if len(img.shape) == 2:
        img = np.stack([img] * 3, axis=2)
    
    # 疊加
    cam_img = (1 - alpha) * img + alpha * heatmap
    cam_img = cam_img / cam_img.max()
    
    return cam_img

# 使用Grad-CAM
# 注意：需要訓練好的模型才能得到有意義的結果
grad_cam = GradCAM(model, model.conv3)

# 生成CAM
model.eval()
cam, pred_class = grad_cam(image)

# 可視化
fig, axes = plt.subplots(1, 3, figsize=(15, 5))

# 原始圖像
original_img = image.cpu().squeeze().numpy()
axes[0].imshow(original_img, cmap='gray')
axes[0].set_title('原始圖像')
axes[0].axis('off')

# 熱力圖
axes[1].imshow(cam, cmap='jet')
axes[1].set_title(f'Grad-CAM (預測類別: {pred_class})')
axes[1].axis('off')

# 疊加圖
cam_img = show_cam_on_image(original_img, cam)
axes[2].imshow(cam_img)
axes[2].set_title('疊加可視化')
axes[2].axis('off')

plt.tight_layout()
plt.show()

print("注意：由於模型未訓練，Grad-CAM結果可能不理想。")
print("在實際應用中，請使用訓練好的模型。")

## 4. 特徵空間可視化

使用降維技術（如t-SNE、UMAP）將高維特徵投影到2D空間，觀察：
- 不同類別的特徵分布
- 類別的可分離性
- 是否存在聚類結構

In [ ]:
def extract_features(model, data_loader, device, max_samples=1000):
    """
    提取模型的特徵向量
    
    Args:
        model: CNN模型
        data_loader: 數據加載器
        device: 設備
        max_samples: 最大樣本數
    """
    model.eval()
    features = []
    labels = []
    
    # 註冊hook獲取倒數第二層的輸出
    activation = {}
    def get_activation(name):
        def hook(model, input, output):
            activation[name] = output.detach()
        return hook
    
    # 為倒數第二個全連接層註冊hook
    handle = model.fc1.register_forward_hook(get_activation('fc1'))
    
    with torch.no_grad():
        for i, (data, target) in enumerate(data_loader):
            if len(features) >= max_samples:
                break
            
            data = data.to(device)
            _ = model(data)
            
            features.append(activation['fc1'].cpu().numpy())
            labels.append(target.numpy())
    
    handle.remove()
    
    features = np.vstack(features)
    labels = np.concatenate(labels)
    
    return features, labels

def visualize_features_tsne(features, labels, n_classes=10):
    """
    使用t-SNE可視化特徵空間
    
    Args:
        features: 特徵向量
        labels: 標籤
        n_classes: 類別數量
    """
    print("執行t-SNE降維...")
    tsne = TSNE(n_components=2, random_state=42, perplexity=30)
    features_2d = tsne.fit_transform(features)
    
    # 繪製散點圖
    plt.figure(figsize=(12, 10))
    
    # 為每個類別使用不同的顏色
    colors = plt.cm.tab10(np.linspace(0, 1, n_classes))
    
    for class_idx in range(n_classes):
        mask = labels == class_idx
        plt.scatter(features_2d[mask, 0], features_2d[mask, 1],
                   c=[colors[class_idx]], label=f'類別 {class_idx}',
                   alpha=0.6, edgecolors='w', linewidth=0.5)
    
    plt.xlabel('t-SNE 維度 1', fontsize=12)
    plt.ylabel('t-SNE 維度 2', fontsize=12)
    plt.title('特徵空間的t-SNE可視化', fontsize=14)
    plt.legend(loc='best', ncol=2)
    plt.grid(True, alpha=0.3)
    plt.tight_layout()
    plt.show()

# 創建數據加載器
test_dataset = FashionMNIST(root='./data', train=False, download=True,
                           transform=transforms.ToTensor())
test_loader = DataLoader(test_dataset, batch_size=64, shuffle=False)

# 提取特徵
print("提取特徵向量...")
features, labels = extract_features(model, test_loader, device, max_samples=1000)
print(f"特徵形狀: {features.shape}")
print(f"標籤形狀: {labels.shape}")

# 可視化
visualize_features_tsne(features, labels)

print("\n注意：未訓練的模型特徵分布可能較為混亂。")
print("訓練後的模型應該會顯示出更清晰的類別聚類。")

## 5. 訓練過程可視化

監控和可視化訓練過程對於理解模型行為至關重要。

In [ ]:
def plot_training_history(history):
    """
    繪製訓練歷史
    
    Args:
        history: 字典，包含'train_loss', 'val_loss', 'train_acc', 'val_acc'
    """
    fig, axes = plt.subplots(1, 2, figsize=(15, 5))
    
    # 損失曲線
    axes[0].plot(history['train_loss'], label='訓練損失', linewidth=2)
    axes[0].plot(history['val_loss'], label='驗證損失', linewidth=2)
    axes[0].set_xlabel('Epoch', fontsize=12)
    axes[0].set_ylabel('損失', fontsize=12)
    axes[0].set_title('訓練和驗證損失', fontsize=14)
    axes[0].legend()
    axes[0].grid(True, alpha=0.3)
    
    # 準確率曲線
    axes[1].plot(history['train_acc'], label='訓練準確率', linewidth=2)
    axes[1].plot(history['val_acc'], label='驗證準確率', linewidth=2)
    axes[1].set_xlabel('Epoch', fontsize=12)
    axes[1].set_ylabel('準確率 (%)', fontsize=12)
    axes[1].set_title('訓練和驗證準確率', fontsize=14)
    axes[1].legend()
    axes[1].grid(True, alpha=0.3)
    
    plt.tight_layout()
    plt.show()

# 示例訓練歷史
example_history = {
    'train_loss': [2.3, 1.8, 1.4, 1.1, 0.9, 0.7, 0.6, 0.5, 0.4, 0.35],
    'val_loss': [2.2, 1.7, 1.3, 1.0, 0.9, 0.8, 0.75, 0.73, 0.72, 0.71],
    'train_acc': [20, 35, 48, 60, 68, 75, 80, 84, 87, 89],
    'val_acc': [22, 38, 50, 62, 68, 72, 74, 75, 76, 76.5]
}

plot_training_history(example_history)

print("\n從曲線可以看出：")
print("- 訓練損失持續下降，表明模型在學習")
print("- 驗證損失在後期趨於平穩，可能接近收斂")
print("- 訓練和驗證準確率的差距逐漸增大，可能存在輕微過擬合")

## 6. 混淆矩陣可視化

混淆矩陣幫助我們理解：
- 哪些類別容易被混淆
- 模型在哪些類別上表現較差
- 如何改進模型

In [ ]:
from sklearn.metrics import confusion_matrix

def plot_confusion_matrix(y_true, y_pred, class_names=None):
    """
    繪製混淆矩陣
    
    Args:
        y_true: 真實標籤
        y_pred: 預測標籤
        class_names: 類別名稱列表
    """
    cm = confusion_matrix(y_true, y_pred)
    
    # 歸一化
    cm_norm = cm.astype('float') / cm.sum(axis=1)[:, np.newaxis]
    
    plt.figure(figsize=(12, 10))
    sns.heatmap(cm_norm, annot=True, fmt='.2f', cmap='Blues',
               xticklabels=class_names if class_names else range(len(cm)),
               yticklabels=class_names if class_names else range(len(cm)))
    
    plt.xlabel('預測標籤', fontsize=12)
    plt.ylabel('真實標籤', fontsize=12)
    plt.title('混淆矩陣（歸一化）', fontsize=14)
    plt.tight_layout()
    plt.show()

# Fashion-MNIST類別名稱
fashion_mnist_classes = ['T-shirt/top', 'Trouser', 'Pullover', 'Dress', 'Coat',
                        'Sandal', 'Shirt', 'Sneaker', 'Bag', 'Ankle boot']

# 生成示例預測（實際應用中應使用真實的預測結果）
np.random.seed(42)
y_true = np.random.randint(0, 10, 1000)
y_pred = y_true.copy()
# 添加一些錯誤
error_indices = np.random.choice(1000, 200, replace=False)
y_pred[error_indices] = np.random.randint(0, 10, 200)

plot_confusion_matrix(y_true, y_pred, fashion_mnist_classes)

print("\n從混淆矩陣可以看出：")
print("- 對角線上的值越高越好（正確預測）")
print("- 非對角線的值顯示了類別間的混淆情況")
print("- 可以針對性地改進容易混淆的類別")

## 7. 錯誤分析可視化

分析模型預測錯誤的樣本可以幫助我們：
- 發現數據中的問題
- 理解模型的弱點
- 指導模型改進方向

In [ ]:
def visualize_predictions(model, data_loader, device, n_samples=16, show_errors_only=False):
    """
    可視化模型預測結果
    
    Args:
        model: CNN模型
        data_loader: 數據加載器
        device: 設備
        n_samples: 顯示的樣本數
        show_errors_only: 是否只顯示錯誤預測
    """
    model.eval()
    images_list = []
    labels_list = []
    preds_list = []
    probs_list = []
    
    with torch.no_grad():
        for data, target in data_loader:
            data = data.to(device)
            output = model(data)
            probs = F.softmax(output, dim=1)
            preds = output.argmax(dim=1)
            
            for i in range(len(data)):
                if show_errors_only and preds[i] == target[i]:
                    continue
                
                images_list.append(data[i].cpu())
                labels_list.append(target[i].item())
                preds_list.append(preds[i].item())
                probs_list.append(probs[i].cpu().numpy())
                
                if len(images_list) >= n_samples:
                    break
            
            if len(images_list) >= n_samples:
                break
    
    # 繪製結果
    n_cols = 4
    n_rows = (n_samples + n_cols - 1) // n_cols
    
    fig, axes = plt.subplots(n_rows, n_cols, figsize=(12, 3*n_rows))
    axes = axes.ravel()
    
    for i in range(min(n_samples, len(images_list))):
        img = images_list[i].squeeze()
        label = labels_list[i]
        pred = preds_list[i]
        prob = probs_list[i][pred]
        
        axes[i].imshow(img, cmap='gray')
        
        # 設置標題顏色（正確：綠色，錯誤：紅色）
        color = 'green' if label == pred else 'red'
        axes[i].set_title(f'真實: {label}\n預測: {pred} ({prob:.2f})', color=color)
        axes[i].axis('off')
    
    # 隱藏多餘的子圖
    for i in range(len(images_list), len(axes)):
        axes[i].axis('off')
    
    title = '錯誤預測樣本' if show_errors_only else '預測結果'
    plt.suptitle(title, fontsize=14, y=1.0)
    plt.tight_layout()
    plt.show()

# 可視化預測結果
print("可視化正確和錯誤的預測：")
visualize_predictions(model, test_loader, device, n_samples=16, show_errors_only=False)

print("\n注意：由於模型未訓練，預測結果可能都是錯誤的。")

## 練習題

### 1. 特徵圖演化分析
訓練一個模型，在訓練的不同階段（epoch 1, 5, 10）保存模型，並可視化同一張圖像在不同階段的特徵圖變化。

### 2. 卷積核演化分析
記錄訓練過程中卷積核的變化，分析哪些卷積核學習到了有意義的模式。

### 3. Grad-CAM對比分析
對於錯誤預測的樣本，使用Grad-CAM分析模型關注的區域是否合理。

### 4. 多層Grad-CAM
實現對不同卷積層的Grad-CAM可視化，比較淺層和深層的注意力區域差異。

### 5. 特徵空間分析
使用t-SNE或UMAP比較訓練前後的特徵空間變化，觀察類別的可分離性如何提升。

### 6. 自定義可視化工具
設計一個交互式可視化工具，可以：
- 選擇任意圖像
- 查看預測結果
- 顯示Grad-CAM
- 展示置信度分布

## 總結

### 可視化技術選擇指南

| 目的 | 推薦技術 |
|------|----------|
| 理解學習到的特徵 | 特徵圖、卷積核可視化 |
| 解釋模型決策 | Grad-CAM、注意力機制 |
| 診斷模型問題 | 混淆矩陣、錯誤分析 |
| 評估特徵質量 | t-SNE、UMAP降維可視化 |
| 監控訓練過程 | 損失曲線、準確率曲線 |

### 實踐建議

1. **多角度分析**：結合多種可視化技術
2. **關注異常**：重點分析錯誤樣本和異常模式
3. **迭代改進**：根據可視化結果調整模型
4. **記錄發現**：保存重要的可視化結果
5. **對比實驗**：比較不同模型或參數的可視化結果

### 工具推薦

- **TensorBoard**：訓練過程監控
- **Weights & Biases**：實驗管理和可視化
- **Netron**：模型架構可視化
- **Captum**：PyTorch模型解釋工具
- **Grad-CAM++**：改進的類激活映射

### 下一步學習

1. 更多解釋技術（LIME, SHAP）
2. 注意力機制可視化
3. 對抗樣本分析
4. 模型壓縮和加速
5. 部署和生產環境可視化